# White Friday Deals - amazon.eg
شغّل الخلايا بالترتيب من فوق لتحت.

## 1) Imports

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
import pandas as pd
import time

## 2) Browser settings

In [ ]:
options = Options()
options.page_load_strategy = "eager"
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("--window-size=1920,1080")

## 3) Open the page

In [ ]:
START_URL = ("https://www.amazon.eg/whitefriday?language=en&_encoding=UTF8"
             "&discounts-widget=%2522%257B%255C%2522state%255C%2522%253A%257B%255C%2522refinementFilters%255C%2522%253A%257B%255C%2522departments%255C%2522%253A%255B%255C%252218018103031%255C%2522%255D%257D%257D%252C%255C%2522version%255C%2522%253A1%257D%2522")

driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(60)

try:
    driver.get(START_URL)
except TimeoutException:
    driver.execute_script("window.stop();")

time.sleep(5)
print("Title:", driver.title)

## 4) Functions
- `collect()` reads the products currently on the page
- `scroll_and_collect()` scrolls slowly and collects at every step
- `click_view_more()` clicks the **View more deals** button (only that one)

In [ ]:
Deals = []
seen_links = set()

JS_COLLECT = r"""
const out = [], seen = new Set();
document.querySelectorAll('a[href*="/dp/"], a[href*="/gp/product/"]').forEach(a => {
  const m = a.href.match(/\/(?:dp|gp\/product)\/([A-Z0-9]{10})/);
  if (!m || seen.has(m[1])) return;
  let c = a, card = null;
  for (let i = 0; i < 8 && c.parentElement; i++) {
    c = c.parentElement;
    if (c.querySelector('.a-price, [class*="rice"]')) { card = c; break; }
  }
  if (!card) return;
  seen.add(m[1]);
  const txt = card.innerText || '';
  const img = card.querySelector('img');
  const price = card.querySelector('.a-price .a-offscreen, .a-price-whole');
  const old = card.querySelector('.a-text-price .a-offscreen, [data-a-strike="true"] .a-offscreen');
  const disc = txt.match(/(\d{1,2})\s*%/);
  out.push({asin: m[1], link: a.href,
            name: (img && img.alt) || a.getAttribute('aria-label') || a.innerText.trim(),
            image: img ? img.src : '',
            price: price ? price.textContent.trim() : '',
            old_price: old ? old.textContent.trim() : '',
            discount: disc ? disc[1] : ''});
});
return out;
"""

JS_FIND_BUTTON = r"""
const re = /^\s*view\s+more\s+deals\s*$/i;
const hits = Array.from(document.querySelectorAll('button, a, span, div, [role=button]'))
  .filter(e => e.offsetParent !== null && re.test(e.innerText || ''));
const deepest = hits.filter(e => !hits.some(o => o !== e && e.contains(o)));
return deepest.map(e => e.closest('button, a, [role=button]') || e);
"""

def collect():
    for item in driver.execute_script(JS_COLLECT):
        link = item["link"].split("?")[0]
        if link in seen_links:
            continue
        seen_links.add(link)
        Deals.append({
            "Name": item["name"],
            "Link": link,
            "Image": item["image"],
            "Price": item["price"] or "Not Available",
            "Old Price": item["old_price"] or "Not Available",
            "Discount %": item["discount"] or "Not Available",
            "ASIN": item["asin"],
        })

def scroll_and_collect(step=400, pause=0.8):
    stuck = 0
    while stuck < 3:
        collect()
        driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(pause)
        at_bottom = driver.execute_script(
            "return window.innerHeight + window.scrollY >= document.body.scrollHeight - 50;")
        stuck = stuck + 1 if at_bottom else 0
    collect()

def click_view_more():
    buttons = driver.execute_script(JS_FIND_BUTTON)
    if not buttons:
        return False
    button = buttons[-1]
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", button)
    return True

## 5) Run: scroll + collect, click the button, repeat

In [ ]:
round_no = 1
failed = 0

while True:
    scroll_and_collect()
    print(f"Round {round_no}: total collected = {len(Deals)}")
    pd.DataFrame(Deals).to_csv("amazon_eg_whitefriday.csv", index=False, encoding="utf-8-sig")

    before = len(Deals)
    if not click_view_more():
        print("No 'View more deals' button left. Done.")
        break

    print("Clicked: View more deals")
    time.sleep(5)
    scroll_and_collect()

    failed = failed + 1 if len(Deals) == before else 0
    if failed >= 2:
        print("No new products after clicking. Stopping.")
        break
    round_no += 1

## 6) Close the browser and show the data

In [ ]:
driver.quit()

df = pd.DataFrame(Deals)
df.to_csv("amazon_eg_whitefriday.csv", index=False, encoding="utf-8-sig")
print(df.shape)
df.head()